In [1]:
import pandas as pd
import geopandas as gpd
import osmnx as ox
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

In [2]:
gdf = gpd.read_file('../data/aa_worda_boundary.zip')

In [3]:
print("Shape of data:", gdf.shape)
print("\nColumn names and types:")
print(gdf.dtypes)
print("\nGemoetry type:")
print(gdf.geometry.type.unique())
print("\nCRS (Coordinate Reference System):")
print(gdf.crs)
print("\nBasic Statistics:")
print(gdf.describe())

Shape of data: (116, 8)

Column names and types:
FID_1          float64
OBJECTID       float64
Sub_City           str
Woreda             str
Shape_Le_1     float64
Shape_Area     float64
Region             str
geometry      geometry
dtype: object

Gemoetry type:
<StringArray>
['Polygon']
Length: 1, dtype: str

CRS (Coordinate Reference System):
EPSG:4326

Basic Statistics:
            FID_1    OBJECTID    Shape_Le_1    Shape_Area
count  116.000000  116.000000    116.000000  1.160000e+02
mean    57.500000   58.500000   9969.958165  4.593159e+06
std     33.630343   33.630343   6485.900215  6.249320e+06
min      0.000000    1.000000   2578.359956  3.148340e+05
25%     28.750000   29.750000   5290.083853  1.101869e+06
50%     57.500000   58.500000   8022.485103  1.993762e+06
75%     86.250000   87.250000  12244.374837  5.078316e+06
max    115.000000  116.000000  32627.357073  2.917746e+07


In [4]:
pd.set_option('display.max_rows', 150)

In [5]:
gdf = gdf.to_crs('EPSG:4326') 
gdf['centroid'] = gdf.geometry.centroid
gdf['lon'] = gdf['centroid'].x
gdf['lat'] = gdf['centroid'].y

In [6]:
gdf.drop(columns=['FID_1', 'OBJECTID', 'Shape_Le_1',	'Shape_Area', 'Region'], inplace=True)
gdf.rename(columns={'Sub_City': 'subcity'}, inplace=True)
gdf.rename(columns={'Woreda': 'district'}, inplace=True)

In [7]:
tags = {'amenity': ['school', 'hospital', 'bank', 'restaurant', 'cafe', 'pharmacy', 'police', 'fuel', 'parking', 'university', 'library', 'atm']}

radii = [500, 1000, 2000, 3000, 4000, 5000]

for radius in radii:
    radius_km = radius/1000
    gdf[f'amenities_within_{radius_km}km'] = 0

In [8]:
for index, row in gdf.iterrows():
    lat = row['lat']
    lon = row['lon']
    
    radius_counts = []
    
    for radius in radii:
        try:
            pois = ox.features_from_point((lat, lon), tags=tags, dist=radius)
            count = len(pois)
        except Exception as e:
            count = 0
        
        radius_counts.append(count)
    for i, radius in enumerate(radii):
        radius_km = radius/1000
        gdf.at[index, f'amenities_within_{radius_km}km'] = radius_counts[i]
    
    print(f"Subcity: {row['subcity']}, District: {row['district']} - "
          f"Counts: {', '.join([f'{r/1000}km:{c}' for r, c in zip(radii, radius_counts)])}")

Subcity: Addis Ketema, District: 10 - Counts: 0.5km:10, 1.0km:38, 2.0km:140, 3.0km:260, 4.0km:451, 5.0km:780
Subcity: Addis Ketema, District: 05 - Counts: 0.5km:14, 1.0km:64, 2.0km:187, 3.0km:318, 4.0km:670, 5.0km:977
Subcity: Addis Ketema, District: 06 - Counts: 0.5km:8, 1.0km:80, 2.0km:251, 3.0km:538, 4.0km:891, 5.0km:1188
Subcity: Addis Ketema, District: 07 - Counts: 0.5km:22, 1.0km:77, 2.0km:283, 3.0km:655, 4.0km:971, 5.0km:1290
Subcity: Addis Ketema, District: 01 - Counts: 0.5km:25, 1.0km:87, 2.0km:442, 3.0km:768, 4.0km:1132, 5.0km:1511
Subcity: Addis Ketema, District: 09 - Counts: 0.5km:23, 1.0km:59, 2.0km:204, 3.0km:483, 4.0km:775, 5.0km:1131
Subcity: Addis Ketema, District: 08 - Counts: 0.5km:32, 1.0km:80, 2.0km:263, 3.0km:622, 4.0km:969, 5.0km:1293
Subcity: Addis Ketema, District: 02 - Counts: 0.5km:16, 1.0km:63, 2.0km:270, 3.0km:616, 4.0km:980, 5.0km:1350
Subcity: Addis Ketema, District: 03 - Counts: 0.5km:3, 1.0km:21, 2.0km:185, 3.0km:442, 4.0km:809, 5.0km:1202
Subcity: Addi

In [16]:
road_types = ['primary', 'secondary', 'tertiary', 'unclassified', 'residential']
tags = {'highway': road_types}

for index, row in gdf.iterrows():
    lat = row['lat']
    lon = row['lon']
    
    radius_counts = []
    
    for radius in radii:
        try:
            # Ensure both lines below have the SAME indentation (4 spaces)
            roads = ox.features_from_point((lat, lon), tags=tags, dist=radius)
            count = len(roads)
        except Exception as e:
            print(f"Error at radius {radius}: {e}")
            count = 0
        
        radius_counts.append(count)
    
    for i, radius in enumerate(radii):
        radius_km = radius / 1000
        gdf.at[index, f'roads_within_{radius_km}km'] = radius_counts[i]
    
    counts_str = ', '.join([f'{r/1000}km:{c}' for r, c in zip(radii, radius_counts)])
    print(f"Subcity: {row['subcity']}, District: {row['district']} - "
          f"Road Counts: {counts_str}")

Subcity: Addis Ketema, District: 10 - Road Counts: 0.5km:155, 1.0km:589, 2.0km:2311, 3.0km:5129, 4.0km:8077, 5.0km:11493
Subcity: Addis Ketema, District: 05 - Road Counts: 0.5km:154, 1.0km:549, 2.0km:2459, 3.0km:5278, 4.0km:7988, 5.0km:11177
Subcity: Addis Ketema, District: 06 - Road Counts: 0.5km:164, 1.0km:622, 2.0km:2524, 3.0km:5284, 4.0km:8075, 5.0km:11421
Subcity: Addis Ketema, District: 07 - Road Counts: 0.5km:219, 1.0km:685, 2.0km:2576, 3.0km:5137, 4.0km:8428, 5.0km:12354
Subcity: Addis Ketema, District: 01 - Road Counts: 0.5km:215, 1.0km:742, 2.0km:2543, 3.0km:4737, 4.0km:8291, 5.0km:12971
Subcity: Addis Ketema, District: 09 - Road Counts: 0.5km:161, 1.0km:654, 2.0km:2456, 3.0km:5421, 4.0km:8751, 5.0km:12220
Subcity: Addis Ketema, District: 08 - Road Counts: 0.5km:198, 1.0km:669, 2.0km:2499, 3.0km:5043, 4.0km:8539, 5.0km:12892
Subcity: Addis Ketema, District: 02 - Road Counts: 0.5km:198, 1.0km:729, 2.0km:2478, 3.0km:4857, 4.0km:8497, 5.0km:13460
Subcity: Addis Ketema, District:

In [17]:
gdf.to_csv('../data/final_feature.csv', index=False)